In [23]:
# dependencies
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.nn as nn
import math


# Rishis notebook

In [9]:
df = pd.read_csv(
    "LD2011_2014.txt",
    sep=";",              # delimiter
    decimal=",",          # VERY IMPORTANT
    parse_dates=[0],      # first column = datetime
    quotechar='"'         # handles quotes
)
df.head()
df.rename(columns={df.columns[0]: "datetime"}, inplace=True)
df.set_index("datetime", inplace=True)
df.isna().sum()[0]==1
df.columns
df["total_load"] = df.sum(axis=1)
df_hourly = df["total_load"].resample("H").sum().to_frame()
df_hourly.head()
# If df_hourly is a Series, convert it to a DataFrame first
if isinstance(df_hourly, pd.Series):
    df_hourly = df_hourly.to_frame(name="total_load")

# Make sure index is datetime
df_hourly.index = pd.to_datetime(df_hourly.index)

# Create a copy so original stays untouched
df_features = df_hourly.copy()

# Basic calendar/time features
df_features["hour_of_day"] = df_features.index.hour
df_features["day_of_week"] = df_features.index.dayofweek   # Monday=0, Sunday=6
df_features["is_weekend"] = (df_features["day_of_week"] >= 5).astype(int)
df_features["month"] = df_features.index.month

# Optional cyclical encoding
df_features["hour_sin"] = np.sin(2 * np.pi * df_features["hour_of_day"] / 24)
df_features["hour_cos"] = np.cos(2 * np.pi * df_features["hour_of_day"] / 24)

df_features["dayofweek_sin"] = np.sin(2 * np.pi * df_features["day_of_week"] / 7)
df_features["dayofweek_cos"] = np.cos(2 * np.pi * df_features["day_of_week"] / 7)

df_features["month_sin"] = np.sin(2 * np.pi * (df_features["month"] - 1) / 12)
df_features["month_cos"] = np.cos(2 * np.pi * (df_features["month"] - 1) / 12)

print(df_features.head())
print(df_features.columns)

                        total_load  hour_of_day  day_of_week  is_weekend  \
datetime                                                                   
2011-01-01 00:00:00  207058.270272            0            5           1   
2011-01-01 01:00:00  265378.510747            1            5           1   
2011-01-01 02:00:00  263924.219533            2            5           1   
2011-01-01 03:00:00  266306.134264            3            5           1   
2011-01-01 04:00:00  259854.210701            4            5           1   

                     month  hour_sin  hour_cos  dayofweek_sin  dayofweek_cos  \
datetime                                                                       
2011-01-01 00:00:00      1  0.000000  1.000000      -0.974928      -0.222521   
2011-01-01 01:00:00      1  0.258819  0.965926      -0.974928      -0.222521   
2011-01-01 02:00:00      1  0.500000  0.866025      -0.974928      -0.222521   
2011-01-01 03:00:00      1  0.707107  0.707107      -0.974928      

/tmp/ipykernel_18111/1167452158.py:11: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df.isna().sum()[0]==1
/tmp/ipykernel_18111/1167452158.py:14: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_hourly = df["total_load"].resample("H").sum().to_frame()


# normalizing df

In [15]:
def normalize_and_split(df_features, train_end='2013-12-31', val_start='2014-01-01',
                        val_end='2014-06-30', test_start='2014-07-01'):

    from sklearn.preprocessing import StandardScaler

    # Chronological split
    train = df_features[:train_end].copy()
    val = df_features[val_start:val_end].copy()
    test = df_features[test_start:].copy()

    # Scale total_load using only training data
    scaler = StandardScaler()
    scaler.fit(train[['total_load']])

    train['total_load_scaled'] = scaler.transform(train[['total_load']])
    val['total_load_scaled'] = scaler.transform(val[['total_load']])
    test['total_load_scaled'] = scaler.transform(test[['total_load']])

    # Print summary
    print(f"Train: {train.index.min()} to {train.index.max()} ({len(train)} rows)")
    print(f"Val:   {val.index.min()} to {val.index.max()} ({len(val)} rows)")
    print(f"Test:  {test.index.min()} to {test.index.max()} ({len(test)} rows)")

    return train, val, test, scaler

In [16]:
train, val, test, scaler = normalize_and_split(df_features)

Train: 2011-01-01 00:00:00 to 2013-12-31 23:00:00 (26304 rows)
Val:   2014-01-01 00:00:00 to 2014-06-30 23:00:00 (4344 rows)
Test:  2014-07-01 00:00:00 to 2015-01-01 00:00:00 (4417 rows)


create sequences

In [20]:
def create_sequences(data, input_window=168, forecast_horizon=24):
    feature_cols = ['total_load_scaled', 'hour_sin', 'hour_cos',
                    'dayofweek_sin', 'dayofweek_cos',
                    'month_sin', 'month_cos']

    features = data[feature_cols].values
    targets = data['total_load_scaled'].values

    X, y = [], []
    for i in range(len(features) - input_window - forecast_horizon + 1):
        X.append(features[i : i + input_window])
        y.append(targets[i + input_window : i + input_window + forecast_horizon])

    return torch.tensor(np.array(X), dtype=torch.float32), torch.tensor(np.array(y), dtype=torch.float32)

In [21]:
X_train, y_train = create_sequences(train)
X_val, y_val = create_sequences(val)
X_test, y_test = create_sequences(test)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape},   y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}")

X_train: torch.Size([26113, 168, 7]), y_train: torch.Size([26113, 24])
X_val:   torch.Size([4153, 168, 7]),   y_val:   torch.Size([4153, 24])
X_test:  torch.Size([4226, 168, 7]),  y_test:  torch.Size([4226, 24])


Data loader ( batching)

In [24]:


def create_dataloaders(X_train, y_train, X_val, y_val, X_test, y_test, batch_size=64):
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size, shuffle=False)

    # Sanity check
    x_batch, y_batch = next(iter(train_loader))
    print(f"Input batch shape:  {x_batch.shape}")
    print(f"Target batch shape: {y_batch.shape}")

    return train_loader, val_loader, test_loader



In [25]:
train_loader, val_loader, test_loader = create_dataloaders(X_train, y_train, X_val, y_val, X_test, y_test)

Input batch shape:  torch.Size([64, 168, 7])
Target batch shape: torch.Size([64, 24])


persistence baseline( for comparision)

In [26]:
def persistence_baseline(X_test, y_test, scaler):
    # Grab the last 24 hours of total_load_scaled from each input window
    # total_load_scaled is the first column (index 0) in our features
    predictions = X_test[:, -24:, 0]  # last 24 timesteps, load column only
    actuals = y_test

    # Convert back to original scale for interpretable metrics
    pred_original = scaler.inverse_transform(predictions.reshape(-1, 1)).reshape(predictions.shape)
    actual_original = scaler.inverse_transform(actuals.reshape(-1, 1)).reshape(actuals.shape)

    mae = np.mean(np.abs(actual_original - pred_original))
    rmse = np.sqrt(np.mean((actual_original - pred_original) ** 2))
    mape = np.mean(np.abs((actual_original - pred_original) / actual_original)) * 100

    print(f"Persistence Baseline:")
    print(f"  MAE:  {mae:.2f}")
    print(f"  RMSE: {rmse:.2f}")
    print(f"  MAPE: {mape:.2f}%")

    return mae, rmse, mape



In [27]:
baseline_mae, baseline_rmse, baseline_mape = persistence_baseline(X_test, y_test, scaler)

Persistence Baseline:
  MAE:  35421.18
  RMSE: 59571.34
  MAPE: 3.87%


# Transformer

In [39]:


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # shape: [1, max_len, d_model]

        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: [batch, seq_len, d_model]
        x = x + self.pe[:, :x.size(1), :]
        return x

In [40]:
class ElectricityTransformer(nn.Module):
    def __init__(self, input_features=7, d_model=64, nhead=4, num_layers=2,
                 dim_feedforward=128, dropout=0.1, input_window=168, forecast_horizon=24):
        super().__init__()

        self.input_window = input_window
        self.forecast_horizon = forecast_horizon

        # Project input features (7) to d_model dimensions (64)
        self.input_projection = nn.Linear(input_features, d_model)

        # Positional encoding
        self.positional_encoding = PositionalEncoding(d_model, max_len=input_window)

        # Transformer encoder stack
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Output head: maps encoded sequence to 24-step forecast
        self.output_head = nn.Sequential(
            nn.Linear(d_model * input_window, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, forecast_horizon)
        )

    def forward(self, x):
        # x shape: [batch, 168, 7]

        # Step 1: Project 7 features to 64 dimensions
        x = self.input_projection(x)          # [batch, 168, 64]

        # Step 2: Add positional encoding
        x = self.positional_encoding(x)       # [batch, 168, 64]

        # Step 3: Pass through transformer encoder
        x = self.transformer_encoder(x)       # [batch, 168, 64]

        # Step 4: Flatten and predict 24 steps
        x = x.reshape(x.size(0), -1)          # [batch, 168 * 64]
        x = self.output_head(x)               # [batch, 24]

        return x

In [41]:
class QuantileLoss(nn.Module):
    def __init__(self, quantiles=[0.1, 0.5, 0.9]):
        super().__init__()
        self.quantiles = quantiles

    def forward(self, predictions, targets):
        # predictions shape: [batch, 24, num_quantiles]
        # targets shape: [batch, 24]

        targets = targets.unsqueeze(-1)  # [batch, 24, 1] for broadcasting
        losses = []

        for i, q in enumerate(self.quantiles):
            errors = targets - predictions[:, :, i].unsqueeze(-1)
            loss = torch.max(q * errors, (q - 1) * errors)
            losses.append(loss.mean())

        return sum(losses) / len(losses)

In [42]:
# Make sure you rerun the updated ElectricityTransformer class first, then:

model = ElectricityTransformer()
criterion = QuantileLoss(quantiles=[0.1, 0.5, 0.9])

test_input = torch.randn(2, 168, 7)
test_target = torch.randn(2, 24)
test_output = model(test_input)

print(f"Output shape: {test_output.shape}")  # verify this says [2, 24, 3]

loss = criterion(test_output, test_target)
print(f"Loss: {loss.item():.4f}")

Output shape: torch.Size([2, 24])


IndexError: too many indices for tensor of dimension 2